# Part I – Reviews Sentiment Classification using TF-IDF

In [ ]:
import nltk
import re
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
 
#required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
data = pd.read_csv('amazon_reviews.csv')
data = data.dropna(subset=['cleaned_review', 'sentiments'])
data['cleaned_review'] = data['cleaned_review'].astype(str)

# Step 1: pre-processing

In [ ]:
stop_words = set(stopwords.words('english'))
 
#keep negation words so sentiment meaning is preserved
words_to_keep = {'not', 'no', 'very'}
custom_stop_words = stop_words - words_to_keep

def preprocess_text(review: str) -> str:
    """Lowercase, expand contractions, remove punctuation,
    tokenize, and strip stop words."""
 
    #1.lowercase
    review = review.lower()
 
    #2.expand common contractions before removing punctuation
    review = review.replace("n't", " not")
    review = review.replace("'re", " are")
    review = review.replace("'ve", " have")
    review = review.replace("'ll", " will")
    review = review.replace("'d",  " would")
    review = review.replace("'m",  " am")
 
    #3.remove punctuation / special characters
    review = re.sub(r'[^\w\s]', '', review)
 
    # 4. Tokenize
    words = word_tokenize(review)
 
    # 5. Remove stop words and non-alpha tokens
    clean_words = [
        word for word in words
        if word not in custom_stop_words and word.isalpha()
    ]
    return " ".join(clean_words)

data['processed_review'] = data['cleaned_review'].apply(preprocess_text)
#drop any rows where preprocessing produced an empty string
data = data[data['processed_review'].str.strip() != '']
data.reset_index(drop=True, inplace=True)

# Step 2: label encoding

In [ ]:
le = LabelEncoder()
data['label_encoded'] = le.fit_transform(data['sentiments'])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Step 3: Data Splitting

In [ ]:
X = data['processed_review']
y = data['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps class proportions equal in both splits
)

# Step 4: TF-IDF VECTORIZATION

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    #apply log normalization to term frequencies
    stop_words=None,      #stop words already removed in preprocessing
)
 
X_train_tfidf = vectorizer.fit_transform(X_train)   #fit+transform on train
X_test_tfidf  = vectorizer.transform(X_test)         #transform only on test

# Step 5: Text classification

In [ ]:
# 5.1 Logistic Regression

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_tfidf, y_train)
y_pred_lr = logistic_model.predict(X_test_tfidf)

In [ ]:
# 5.2: SVM

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

# Step 6 – Classification Reports

In [ ]:
target_names = le.classes_.tolist()   #['Negative', 'Neutral', 'Positive']
 
print("\n" + "=" * 50)
print("          Logistic Regression Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=target_names))
 
print("\n" + "=" * 50)
print("                  SVM Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm, target_names=target_names))
 
#models comparison
lr_acc  = accuracy_score(y_test, y_pred_lr)
svm_acc = accuracy_score(y_test, y_pred_svm)
 
print("\n" + "=" * 50)
print("             Model Comparison")
print("=" * 50)
print(f"Logistic Regression Accuracy : {lr_acc:.4f}")
print(f"SVM Accuracy                 : {svm_acc:.4f}")
 
if lr_acc > svm_acc:
    print(">> Logistic Regression performed better overall.")
elif svm_acc > lr_acc:
    print(">> SVM performed better overall.")
else:
    print(">> Both models performed equally.")

# Step 7(Bonus): Predict the label of new review

In [ ]:
def predict_review(review_text: str, model) -> str:
    """Pre-process a raw review, vectorize it with the SAME
    TF-IDF vectorizer (transform only), and return the predicted label."""
    processed = preprocess_text(review_text)
    vector = vectorizer.transform([processed])
    prediction = model.predict(vector)
    return le.inverse_transform(prediction)[0]

#example
user_input = input("\nEnter a review to classify: ")
 
lr_result  = predict_review(user_input, logistic_model)
svm_result = predict_review(user_input, svm_model)
 
print(f"\nLogistic Regression → {lr_result}")
print(f"SVM→ {svm_result}")

# Positive (2):"This product is amazing! The quality is excellent and it exceeded my expectations. I would definitely buy it again."
# Neutral (1): "The product is okay. It works as described but nothing special. Delivery was fast though."
# Negative (0): "The product stopped working after two days. Very poor quality and a complete waste of money. I am really disappointed." 


---
# Part II – Tweets Emotion Classification using Word Embeddings

In [30]:
import pandas as pd
import numpy as np
import tensorflow as tf
import gensim.downloader as gensim_api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D,
    Flatten, Dense, Input, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [32]:
twdataset = pd.read_csv('twitter_emotion.csv')
print(twdataset.shape)
print(twdataset.head())


(416809, 3)
   Unnamed: 0                                               text  label
0           0      i just feel really helpless and heavy hearted      4
1           1  ive enjoyed being able to slouch about relax a...      0
2           2  i gave up my internship with the dmrg and am f...      4
3           3                         i dont know i feel so lost      0
4           4  i am a kindergarten teacher and i am thoroughl...      4


In [33]:
TWTEXTCOL  = 'text'
TWLABELCOL = 'label'
print(f'Text column : {TWTEXTCOL}')
print(f'Label column: {TWLABELCOL}')
print('\nEmotion distribution:')
print(twdataset[TWLABELCOL].value_counts())


Text column : text
Label column: label

Emotion distribution:
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


## 1. Tweets Pre-processing – Keras Tokenizer

In [34]:
MAX_WORDS   = 100_000
MAX_SEQ_LEN = 60   # Tweets are short; 60 covers ~99th percentile of tweet lengths

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(twdataset[TWTEXTCOL].astype(str))
sequences  = tokenizer.texts_to_sequences(twdataset[TWTEXTCOL].astype(str))
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1   # actual vocab size, not MAX_WORDS+1

print(f'Vocab size        : {len(word_index)}')
print(f'First 10 entries  : {list(word_index.items())[:10]}')

padded_sequences = pad_sequences(sequences,maxlen=MAX_SEQ_LEN)
print(f'Padded sequences shape: {padded_sequences.shape}')

#analysis of raw tweet lengths before padding
raw_lengths = [len(seq) for seq in sequences]
print(f'\nTweet length stats:')
print(f'  Mean   : {np.mean(raw_lengths):.0f} tokens')
print(f'  Median : {np.median(raw_lengths):.0f} tokens')
print(f'  95th % : {np.percentile(raw_lengths, 95):.0f} tokens')
print(f'  99th % : {np.percentile(raw_lengths, 99):.0f} tokens')
print(f'  Max    : {max(raw_lengths)} tokens')


Vocab size        : 75302
First 10 entries  : [('i', 1), ('feel', 2), ('and', 3), ('to', 4), ('the', 5), ('a', 6), ('feeling', 7), ('that', 8), ('of', 9), ('my', 10)]
Padded sequences shape: (416809, 60)

Tweet length stats:
  Mean   : 19 tokens
  Median : 17 tokens
  95th % : 41 tokens
  99th % : 52 tokens
  Max    : 178 tokens


## 2a. Pre-trained Embeddings – GloVe Twitter 50d

In [35]:
glove_model = gensim_api.load('glove-twitter-50')
EMBEDDING_DIM_GLOVE = 50
print(f'Vocabulary size in GloVe model: {len(glove_model.key_to_index)}')

Vocabulary size in GloVe model: 1193514


In [36]:
def buildEmbeddingMatrix(wvModel, word_index, vocab_size, embedding_dimension):
    # Matrix sized to actual vocab_size (not MAX_WORDS+1)
    matrix = np.zeros((vocab_size, embedding_dimension))
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        try:
            matrix[idx] = wvModel[word]
        except KeyError:
            pass
    return matrix

glove_embedding_matrix = buildEmbeddingMatrix(glove_model, word_index, vocab_size, EMBEDDING_DIM_GLOVE)
print(f'GloVe embedding matrix shape: {glove_embedding_matrix.shape}')


GloVe embedding matrix shape: (75303, 50)


## 2b. Your trained word2vec model word embeddings:

In [37]:
print("Step 2b – Training custom Word2Vec model ...")

# Tokenize directly from raw tweet text
tokenized_tweets = [
    simple_preprocess(str(tweet))
    for tweet in twdataset[TWTEXTCOL]
]

print(f"Total tokenized tweets : {len(tokenized_tweets)}")
print(f"First tokenized tweet  : {tokenized_tweets[0]}")

EMBEDDING_DIM_W2V = 50   # Match GloVe dimension for fair comparison

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=EMBEDDING_DIM_W2V,
    window=3,       # smaller window suits short tweets
    min_count=2,    # drop very rare words (noise)
    epochs=3,      # more training epochs
    sg=0,           # CBOW: better for short documents
    workers=4
)

w2v_model.save("word2vec_twitter.model")
print(f"Word2Vec vocab size : {len(w2v_model.wv.key_to_index)}")

print("Building Word2Vec embedding matrix ...")
w2v_embedding_matrix = buildEmbeddingMatrix(
    w2v_model.wv, word_index, vocab_size, EMBEDDING_DIM_W2V
)
print(f"Word2Vec embedding matrix shape: {w2v_embedding_matrix.shape}")


Step 2b – Training custom Word2Vec model ...
Total tokenized tweets : 416809
First tokenized tweet  : ['just', 'feel', 'really', 'helpless', 'and', 'heavy', 'hearted']


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec vocab size : 40534
Building Word2Vec embedding matrix ...
Word2Vec embedding matrix shape: (75303, 50)


## 3. Data preparation and splitting:

3a. Apply one-hot encoding for the integer labels using keras.

In [ ]:
labels = twdataset[TWLABELCOL].values.astype(int)   
one_hot_labels = to_categorical(labels, num_classes=6)

print("Original labels shape:", labels.shape)
print("One-hot labels shape:", one_hot_labels.shape)


Original labels shape: (416809,)
One-hot labels shape: (416809, 6)


## 3b. Splitting


In [39]:
# split 80% & 20%
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("\n80/20 Split:")
print("X_train shape:", X_train_80.shape)
print("X_test shape :", X_test_80.shape)

# split 70% & 30%
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

print("\n70/30 Split:")
print("X_train shape:", X_train_70.shape)
print("X_test shape :", X_test_70.shape)



80/20 Split:
X_train shape: (333447, 60)
X_test shape : (83362, 60)

70/30 Split:
X_train shape: (291766, 60)
X_test shape : (125043, 60)


## 4.1 — Model Builder Function

In [40]:
def build_cnn_model(embedding_matrix, embedding_dim, max_len, name="CNN",trainable_embeddings=False):

    vocab_size_local = embedding_matrix.shape[0]
    sequence_input   = Input(shape=(max_len,), dtype='int32')

    embedding_layer = Embedding(
        vocab_size_local,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=trainable_embeddings
    )
    x = embedding_layer(sequence_input)

    # 1st Conv1D 
    x = Conv1D(128, 5, activation='relu', padding='same')(x)
    x = MaxPooling1D(5)(x)   

    # 2nd Conv1D 
    x = Conv1D(128, 5, activation='relu', padding='same')(x)
    x = MaxPooling1D(5)(x)

    x = GlobalMaxPooling1D()(x)
    x = Dense(128, activation='relu')(x)

    output_layer = Dense(6, activation='softmax')(x)

    model = Model(sequence_input, output_layer, name=name)
    model.compile(
        loss='categorical_crossentropy',
        optimizer='rmsprop',
        metrics=['accuracy']
    )
    return model

## 4.2 — Training & Plotting Helpers

In [41]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test,epochs=3, batch_size=256):

    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )
    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n★ Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}\n")
    return history, test_loss, test_acc



## 4.3 — Run All 4 Experiments & print comparison

In [ ]:
all_results = {}  #accumulates all experiments across results

for MAX_LEN in [500, 700, 1000]:

    print("\n" + "█" * 70)
    print(f"  MAX_LEN = {MAX_LEN}")
    print("█" * 70)
    padded = pad_sequences(sequences, maxlen=MAX_LEN,padding='post', truncating='post')

    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
        padded,
        one_hot_labels,
        test_size=0.2,
        random_state=42, 
        stratify=labels
    )
    X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
        padded, 
        one_hot_labels, 
        test_size=0.3,
        random_state=42, 
        stratify=labels
    )

    #GloVe - 80/20
    print("=" * 60)
    print(f"  GloVe-50d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_glove_80 = build_cnn_model(
        glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN,
        name="CNN_GloVe_80_20", trainable_embeddings=False
    )
    model_glove_80.summary()
    hist, loss, acc = train_and_evaluate(
        model_glove_80, X_train_80, X_test_80, y_train_80, y_test_80
    )
    all_results[f"GloVe-50d | 80/20 | L={MAX_LEN}"] = {"MAX_LEN": MAX_LEN, "Split": "80/20","Embedding": "GloVe-50d","loss": loss, "accuracy": acc}

    #GloVe - 70/30
    print("=" * 60)
    print(f"  GloVe-50d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_glove_70 = build_cnn_model(
        glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN,
        name="CNN_GloVe_70_30", trainable_embeddings=False
    )
    model_glove_70.summary()
    hist, loss, acc = train_and_evaluate(
        model_glove_70, X_train_70, X_test_70, y_train_70, y_test_70
    )
    all_results[f"GloVe-50d | 70/30 | L={MAX_LEN}"] = {"MAX_LEN": MAX_LEN, "Split": "70/30","Embedding": "GloVe-50d","loss": loss, "accuracy": acc}

#--------------------------------------------------------------------------------------------------------------------------

    #Word2Vec - 80/20
    print("=" * 60)
    print(f"  Word2Vec-{EMBEDDING_DIM_W2V}d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_w2v_80 = build_cnn_model(w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN,name="CNN_W2V_80_20", trainable_embeddings=True)
    model_w2v_80.summary()
    hist, loss, acc = train_and_evaluate(
        model_w2v_80, X_train_80, X_test_80, y_train_80, y_test_80
    )
    all_results[f"Word2Vec | 80/20 | L={MAX_LEN}"] = {"MAX_LEN": MAX_LEN, "Split": "80/20","Embedding": "Word2Vec","loss": loss, "accuracy": acc}

    #Word2Vec - 70/30
    print("=" * 60)
    print(f"  Word2Vec-{EMBEDDING_DIM_W2V}d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_w2v_70 = build_cnn_model(
        w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN,
        name="CNN_W2V_70_30", trainable_embeddings=True
    )
    model_w2v_70.summary()
    hist, loss, acc = train_and_evaluate(
        model_w2v_70, X_train_70, X_test_70, y_train_70, y_test_70
    )
    all_results[f"Word2Vec | 70/30 | L={MAX_LEN}"] = {"MAX_LEN": MAX_LEN, "Split": "70/30", "Embedding": "Word2Vec","loss": loss, "accuracy": acc}

    # ── Per-MAX_LEN CSV snapshot ───────────────────────────────
    snapshot_df = pd.DataFrame(all_results).T
    snapshot_df.to_csv(f"results_maxlen_{MAX_LEN}.csv")
    print(f"\nSaved: results_maxlen_{MAX_LEN}.csv")

# ── FINAL comparison table (outside loop) ─────────────────────
results_df = pd.DataFrame(all_results).T
results_df.index.name = "Experiment"
results_df = results_df.round(4)

print("\n\n" + "=" * 70)
print("  FINAL COMPARISON TABLE — All Models - All MAX_LEN values")
print("=" * 70)
print(results_df.to_string())


pivot = results_df.pivot_table(
    index="Embedding",
    columns=["Split", "MAX_LEN"],
    values="accuracy"
).round(4)
print("\nAccuracy Pivot Table:")
print(pivot.to_string())


██████████████████████████████████████████████████████████████████████
  MAX_LEN = 500
██████████████████████████████████████████████████████████████████████
  GloVe-50d  |  80/20 Split  |  MAX_LEN=500


/Users/applecare/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "CNN_GloVe_80_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 500, 50)        │     3,765,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 500, 128)       │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,896,612 (14.86 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 700)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 700, 50)        │     3,765,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 700, 128)       │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 140, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 140, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 28, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,896,612 (14.86 MB)

 Trainable params: 131,462 (513.52 KB)

 Non-trainable params: 3,765,150 (14.36 MB)

Epoch 1/3
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 338s 288ms/step - accuracy: 0.7241 - loss: 0.7379 - val_accuracy: 0.8391 - val_loss: 0.4225
Epoch 2/3
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 353s 301ms/step - accuracy: 0.8834 - loss: 0.2908 - val_accuracy: 0.8994 - val_loss: 0.2366
Epoch 3/3
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 333s 284ms/step - accuracy: 0.9091 - loss: 0.1997 - val_accuracy: 0.9080 - val_loss: 0.1911


## 5 - classify user input with model of the higher accuracy

In [ ]:
import numpy as np

LABELS = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}
trained_models = {}
for key, val in all_results.items():
    if "GloVe" in key and "80/20" in key:
        model_obj = model_glove_80
    elif "GloVe" in key and "70/30" in key:
        model_obj = model_glove_70
    elif "Word2Vec" in key and "80/20" in key:
        model_obj = model_w2v_80
    elif "Word2Vec" in key and "70/30" in key:
        model_obj = model_w2v_70
    else:
        continue
    trained_models[key] = {"model": model_obj, "accuracy": val["accuracy"]}

#Pick best model
best_key   = max(trained_models, key=lambda k: trained_models[k]["accuracy"])
best_model = trained_models[best_key]["model"]
BEST_MAX_LEN = int(best_key.split("L=")[-1])  

print(f"Best model : {best_key}")
print(f"Accuracy   : {trained_models[best_key]['accuracy']:.4f}")
print(f"MAX_LEN    : {BEST_MAX_LEN}")

# classification function
def classifyUserInput(text, model, tokenizer, max_len, label_map):
    seq    = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    probs      = model.predict(padded, verbose=0)[0]
    pred_idx   = int(np.argmax(probs))
    pred_label = label_map[pred_idx]
    confidence = float(probs[pred_idx])
    breakdown  = {label_map[i]: round(float(p), 4) for i, p in enumerate(probs)}
    return pred_label, confidence, breakdown

#user input
user_input = input("Enter a message to classify: ")

pred_label, confidence, breakdown = classifyUserInput(
    text      = user_input,
    model     = best_model,
    tokenizer = tokenizer,
    max_len   = BEST_MAX_LEN,
    label_map = LABELS
)

#print results
print(f"\nInput     : {user_input}")
print(f"Emotion   : {pred_label.upper()}")
print(f"Confidence: {confidence:.2%}")
print("\nAll scores:")
for emotion, score in sorted(breakdown.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 30)
    print(f"  {emotion:<10} {score:.4f}  {bar}")

Best model : Word2Vec | 70/30
Accuracy   : 0.9322


KeyboardInterrupt: Interrupted by user